In [18]:
import os
from dotenv import load_dotenv
from pathlib import Path
from Classes.model_classes import SQLLineageExtractor

load_dotenv()

GIGACHAT_CREDENTIALS = os.environ.get("GIGACHAT_CREDENTIALS")

PROMPT = """You are a SQL lineage extraction expert. Extract source-to-target lineage from the SQL statement below and return **ONLY** a JSON object with two keys: "target" and "sources".

**SQL Statement:**
{sql_text}

**Extraction Rules:**
1. **Target**: The main object being created/modified (fully qualified name).
2. **Sources**: All base tables/views referenced in the query (fully qualified names, no aliases).
3. Strip all quotes (e.g., "schema"."table" → schema.table).
4. Exclude derived tables, CTEs, system tables.
5. Remove duplicates from sources.
6. Use lowercase for all names.

**Output Format:**
{format_instructions}

**Example:**
SQL: INSERT INTO analytics.sales_summary SELECT p.category, SUM(s.amount) FROM products.raw_data p JOIN sales.transactions s ON p.id = s.product_id;
Output: {{"target": "analytics.sales_summary", "sources": ["products.raw_data", "sales.transactions"]}}

**Your Response (JSON only):**"""

In [19]:
# Get the current working directory
current_dir = Path.cwd()
# Construct the path relative to current directory
file_path = current_dir / 'data' / 'SQL.txt'

# Read file with example
with open(file_path, "r", encoding="utf-8") as f:
    SQL = f.read()

In [20]:
extractor = SQLLineageExtractor(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_API_KEY
    model="GigaChat",
    verify_ssl_certs=False,
    temperature=0.01,
    template=PROMPT
)

In [21]:
lineage = await extractor.extract_lineage(SQL)
lineage

{'target': 's_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred',
 'sources': ['s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_tmp',
  's_grnplm_as_t_didsd_010_vd_dwh.v_$eks_agrmnt_to_coa_3',
  's_grnplm_as_t_didsd_010_vd_dwh.v_coa',
  's_grnplm_vd_t_bvd_db_dmslcl.a_agr_cred_coa_period',
  's_grnplm_vd_t_bvd_db_dmslcl.a_agr_cred_coa_period',
  's_grnplm_vd_t_bvd_db_dmslcl.coa_crncs_sub',
  's_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_optn',
  's_grnplm_as_t_didsd_010_vd_dwh.v_loan_agrmnt_rate',
  's_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_cust',
  's_grnplm_as_t_didsd_029_vd_dwh.v_agr_cred',
  's_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_core_uvdo',
  's_grnplm_as_t_didsd_029_vd_dwh.v_crncy',
  's_grnplm_as_t_didsd_029_vd_dwh.v_agr_cred_metric_hist$$$,\n    ']}